In [0]:
VOLUME_PATH = "/Volumes/ipl2027/main/main"

files = [
    "ball_by_ball_data.csv",
    "ipl_matches_data.csv",
    "players-data-updated.csv",
    "team_aliases.csv",
    "teams_data.csv"
]

for f in files:
    df_tmp = spark.read.option("header", True).option("inferSchema", True).csv(f"{VOLUME_PATH}/{f}")
    print(f"=== {f} ({df_tmp.count()} rows) ===")
    print(df_tmp.columns, "\n")

In [0]:
from pyspark.sql import functions as F

CATALOG = "ipl2027"
SCHEMA = "main"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/main"

spark.sql(f"USE {CATALOG}.{SCHEMA}")

raw_tables = {
    "bronze_ball_by_ball": "ball_by_ball_data.csv",
    "bronze_matches":      "ipl_matches_data.csv",
    "bronze_players":      "players-data-updated.csv",
    "bronze_team_aliases": "team_aliases.csv",
    "bronze_teams":        "teams_data.csv"
}

for table_name, file_name in raw_tables.items():
    df_raw = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{VOLUME_PATH}/{file_name}")
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.lit(file_name))
    )
    # Clean any accidental spaces in column headers
    for col_name in df_raw.columns:
        clean_col = col_name.strip().lower().replace(" ", "_")
        if clean_col != col_name:
            df_raw = df_raw.withColumnRenamed(col_name, clean_col)
            
    df_raw.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    print(f"✅ Created {CATALOG}.{SCHEMA}.{table_name}")

In [0]:
from pyspark.sql import functions as F

spark.sql("USE ipl2027.main")

# 1. Clean Team Aliases Lookup (maps every historical franchise name -> standardized 3-letter code)
df_aliases = spark.table("bronze_team_aliases")
alias_cols = df_aliases.columns  # Typically [alias_name, team_code] or [old_name, new_name]
df_aliases_clean = (
    df_aliases
    .select(
        F.trim(F.col(alias_cols[0])).alias("raw_team_name"),
        F.upper(F.trim(F.col(alias_cols[1]))).alias("standard_team_code")
    )
    .dropDuplicates(["raw_team_name"])
)
df_aliases_clean.write.mode("overwrite").saveAsTable("silver_team_aliases")

# 2. Clean Matches Table (Deduplicate venues + standardize team codes)
df_matches = spark.table("bronze_matches")
m_cols = df_matches.columns
venue_col = "venue" if "venue" in m_cols else [c for c in m_cols if "venue" in c][0]
t1_col = "team1" if "team1" in m_cols else [c for c in m_cols if "team" in c][0]
t2_col = "team2" if "team2" in m_cols else [c for c in m_cols if "team" in c][1]

df_matches_clean = (
    df_matches
    .dropDuplicates(["match_id"])
    .withColumn("venue_clean", F.trim(F.split(F.col(venue_col), ",").getItem(0)))
    .withColumn(
        "pitch_type",
        F.when(F.col("venue_clean").rlike("Chinnaswamy|Wankhede|Eden Gardens|Rajiv Gandhi|Arun Jaitley"), "Batting Paradise")
         .when(F.col("venue_clean").rlike("Chidambaram|Ekana|Sawai Mansingh|Narendra Modi"), "Spin-Friendly")
         .otherwise("Balanced / Seam")
    )
    .join(df_aliases_clean.alias("a1"), F.trim(F.col(t1_col)) == F.col("a1.raw_team_name"), "left")
    .join(df_aliases_clean.alias("a2"), F.trim(F.col(t2_col)) == F.col("a2.raw_team_name"), "left")
    .withColumn("team1_code", F.coalesce(F.col("a1.standard_team_code"), F.col(t1_col)))
    .withColumn("team2_code", F.coalesce(F.col("a2.standard_team_code"), F.col(t2_col)))
    .drop("raw_team_name", "standard_team_code", "_ingested_at", "_source_file")
)
df_matches_clean.write.mode("overwrite").saveAsTable("silver_matches")

# 3. Clean Players Dimension
df_players_clean = (
    spark.table("bronze_players")
    .dropDuplicates()
    .drop("_ingested_at", "_source_file")
)
df_players_clean.write.mode("overwrite").saveAsTable("silver_players")

# 4. Clean Ball-by-Ball Deliveries
df_balls = spark.table("bronze_ball_by_ball")
b_cols = df_balls.columns

# Auto-detect column names across common IPL CSV variants
over_col    = next((c for c in b_cols if c in ["over", "over_num", "over_number"]), "over")
batter_col  = next((c for c in b_cols if c in ["batter", "striker", "batsman"]), "batter")
bowler_col  = next((c for c in b_cols if c in ["bowler"]), "bowler")
bat_run_col = next((c for c in b_cols if c in ["batsman_runs", "runs_off_bat", "batter_runs"]), "batsman_runs")
tot_run_col = next((c for c in b_cols if c in ["total_runs", "runs_total"]), bat_run_col)
bat_tm_col  = next((c for c in b_cols if c in ["batting_team", "team_batting"]), "batting_team")
bowl_tm_col = next((c for c in b_cols if c in ["bowling_team", "team_bowling"]), "bowling_team")
wkt_col     = next((c for c in b_cols if c in ["dismissal_kind", "wicket_type"]), None)
wide_col    = next((c for c in b_cols if "wide" in c), None)
noball_col  = next((c for c in b_cols if "noball" in c or "no_ball" in c), None)

# Normalize 0-indexed vs 1-indexed overs so Powerplay is always 1..6, Middle 7..15, Death 16..20
min_over = df_balls.agg(F.min(F.col(over_col).cast("int"))).collect()[0][0]
over_offset = 1 if min_over == 0 else 0

df_balls_silver = (
    df_balls
    .filter(F.col("inning").isin([1, 2]) if "inning" in b_cols else F.col("innings").isin([1, 2]))
    .withColumn("over_num", F.col(over_col).cast("int") + F.lit(over_offset))
    .withColumn(
        "match_phase",
        F.when(F.col("over_num") <= 6, "Powerplay")
         .when(F.col("over_num") <= 15, "Middle")
         .otherwise("Death")
    )
    .withColumn(
        "is_legal_ball",
        F.when(
            (F.coalesce(F.col(wide_col).cast("int"), F.lit(0)) == 0 if wide_col else F.lit(True)) &
            (F.coalesce(F.col(noball_col).cast("int"), F.lit(0)) == 0 if noball_col else F.lit(True)),
            1
        ).otherwise(0)
    )
    .withColumn(
        "is_bowler_wicket",
        F.when(
            F.col(wkt_col).isin(["bowled", "caught", "lbw", "stumped", "caught and bowled", "hit wicket"]),
            1
        ).otherwise(0) if wkt_col else F.lit(0)
    )
    .join(df_aliases_clean.alias("bat_a"), F.trim(F.col(bat_tm_col)) == F.col("bat_a.raw_team_name"), "left")
    .join(df_aliases_clean.alias("bowl_a"), F.trim(F.col(bowl_tm_col)) == F.col("bowl_a.raw_team_name"), "left")
    .withColumn("batting_team_code", F.coalesce(F.col("bat_a.standard_team_code"), F.col(bat_tm_col)))
    .withColumn("bowling_team_code", F.coalesce(F.col("bowl_a.standard_team_code"), F.col(bowl_tm_col)))
    .select(
        "match_id",
        "over_num",
        "match_phase",
        "batting_team_code",
        "bowling_team_code",
        F.trim(F.col(batter_col)).alias("batter_name"),
        F.trim(F.col(bowler_col)).alias("bowler_name"),
        F.coalesce(F.col(bat_run_col).cast("int"), F.lit(0)).alias("batter_runs"),
        F.coalesce(F.col(tot_run_col).cast("int"), F.lit(0)).alias("total_runs"),
        "is_legal_ball",
        "is_bowler_wicket"
    )
)
df_balls_silver.write.mode("overwrite").saveAsTable("silver_ball_by_ball")
print("✅ Silver layer tables created cleanly!")

In [0]:
from pyspark.sql import functions as F

spark.sql("USE ipl2027.main")

# 1. GOLD DIMENSION: dim_matches
spark.sql("""
CREATE OR REPLACE TABLE dim_matches AS
SELECT * FROM silver_matches
""")

# 2. GOLD DIMENSION: dim_players
spark.sql("""
CREATE OR REPLACE TABLE dim_players AS
SELECT * FROM silver_players
""")

# 3. GOLD DIMENSION: dim_franchises_2027 (Official Nov 2026 Post-Retention Purses & Open Slots)
franchises_2027 = [
    ("LSG",  "Lucknow Super Giants",        47.45, 18, 7, 3, "Middle-Order Power & Overseas Pace"),
    ("KKR",  "Kolkata Knight Riders",       41.65, 17, 8, 3, "Death Bowler & Top-Order Anchor"),
    ("SRH",  "Sunrisers Hyderabad",         40.15, 18, 7, 3, "Explosive Opener & Overseas All-Rounder"),
    ("DC",   "Delhi Capitals",              39.50, 16, 9, 4, "Left-Arm Pace & Top-Order Batter"),
    ("MI",   "Mumbai Indians",              32.60, 16, 9, 2, "Overseas Spinner & Powerplay Pacer"),
    ("CSK",  "Chennai Super Kings",         28.05, 19, 6, 2, "Indian All-Rounder & Death Pacer"),
    ("RR",   "Rajasthan Royals",            27.05, 19, 6, 3, "Lower-Order Finisher & Left-Arm Pacer"),
    ("PBKS", "Punjab Kings",                18.60, 19, 6, 2, "Backup Overseas Fast Bowler"),
    ("RCB",  "Royal Challengers Bengaluru", 14.15, 20, 5, 1, "Indian Domestic Spin Depth"),
    ("GT",   "Gujarat Titans",               8.75, 22, 3, 2, "Budget Domestic Backup Finisher")
]
df_franchises = spark.createDataFrame(
    franchises_2027,
    ["team_code", "team_name", "remaining_purse_cr", "retained_count", "open_slots", "open_overseas_slots", "primary_auction_need"]
)
df_franchises.write.mode("overwrite").saveAsTable("dim_franchises_2027")

# 4. GOLD DIMENSION: dim_auction_pool_2027 (Official Released Players Entering Dec 16, 2026 Auction)
auction_pool_2027 = [
    ("Matheesha Pathirana", "SL",  True,  "Bowler",      "Death Specialist (Sling)",      "KKR",  2.00, 16.50),
    ("Trent Boult",         "NZ",  True,  "Bowler",      "Left-Arm Powerplay Pacer",      "MI",   2.00, 11.00),
    ("Shimron Hetmyer",     "WI",  True,  "Batter",      "Left-Hand Finisher",            "RR",   1.50,  9.50),
    ("Pathum Nissanka",     "SL",  True,  "Batter",      "Right-Hand Top-Order",          "DC",   1.50,  8.75),
    ("Rachin Ravindra",     "NZ",  True,  "All-Rounder", "Left-Arm Spin All-Rounder",     "KKR",  1.50,  8.25),
    ("Anrich Nortje",       "SA",  True,  "Bowler",      "Express Pace",                  "LSG",  2.00,  8.50),
    ("Wanindu Hasaranga",   "SL",  True,  "All-Rounder", "Leg-Spin All-Rounder",          "LSG",  2.00,  9.25),
    ("Lockie Ferguson",     "NZ",  True,  "Bowler",      "Express Pace",                  "PBKS", 2.00,  7.50),
    ("Avesh Khan",          "IND", False, "Bowler",      "Indian Death/Middle Pacer",     "LSG",  1.00,  8.00),
    ("Deepak Chahar",       "IND", False, "Bowler",      "Indian Powerplay Swing",        "MI",   1.00,  7.25),
    ("Nehal Wadhera",       "IND", False, "Batter",      "Indian Left-Hand Middle-Order", "PBKS", 0.75,  6.50),
    ("Shahrukh Khan",       "IND", False, "Batter",      "Indian Power Finisher",         "GT",   0.75,  6.00),
    ("Jacob Bethell",       "ENG", True,  "All-Rounder", "Left-Arm Spin All-Rounder",     "RCB",  1.00,  5.50)
]
df_auction = spark.createDataFrame(
    auction_pool_2027,
    ["player_name", "country", "is_overseas", "role", "skill_style", "prev_team_code", "base_price_cr", "projected_price_cr"]
)
df_auction.write.mode("overwrite").saveAsTable("dim_auction_pool_2027")

# 5. GOLD FACT TABLE: fact_ball_by_ball
spark.sql("""
CREATE OR REPLACE TABLE fact_ball_by_ball AS
SELECT * FROM silver_ball_by_ball
""")

# 6. GOLD AGGREGATED VIEWS (Pre-computed T20 Metrics for Zero-Hallucination Genie Queries)
spark.sql("""
CREATE OR REPLACE VIEW gold_bowler_phase_metrics AS
SELECT
    b.bowler_name,
    a.country,
    a.is_overseas,
    a.skill_style,
    a.prev_team_code,
    a.base_price_cr,
    a.projected_price_cr,
    m.pitch_type,
    b.match_phase,
    SUM(b.is_legal_ball) AS legal_balls_bowled,
    SUM(b.total_runs) AS runs_conceded,
    SUM(b.is_bowler_wicket) AS wickets_taken,
    ROUND(SUM(b.total_runs) * 6.0 / NULLIF(SUM(b.is_legal_ball), 0), 2) AS economy_rate,
    ROUND(SUM(b.is_legal_ball) * 1.0 / NULLIF(SUM(b.is_bowler_wicket), 0), 2) AS bowling_strike_rate
FROM fact_ball_by_ball b
JOIN dim_matches m ON b.match_id = m.match_id
LEFT JOIN dim_auction_pool_2027 a ON b.bowler_name = a.player_name
GROUP BY ALL
""")

spark.sql("""
CREATE OR REPLACE VIEW gold_batter_phase_metrics AS
SELECT
    b.batter_name,
    a.country,
    a.is_overseas,
    a.skill_style,
    a.prev_team_code,
    a.base_price_cr,
    a.projected_price_cr,
    m.pitch_type,
    b.match_phase,
    SUM(b.is_legal_ball) AS legal_balls_faced,
    SUM(b.batter_runs) AS total_runs_scored,
    ROUND(SUM(b.batter_runs) * 100.0 / NULLIF(SUM(b.is_legal_ball), 0), 2) AS strike_rate,
    SUM(CASE WHEN b.batter_runs IN (4, 6) THEN 1 ELSE 0 END) AS boundary_count
FROM fact_ball_by_ball b
JOIN dim_matches m ON b.match_id = m.match_id
LEFT JOIN dim_auction_pool_2027 a ON b.batter_name = a.player_name
GROUP BY ALL
""")

# 7. ADD PRIMARY & FOREIGN KEYS + COMMENTS (Genie Ontology Grounding)
spark.sql("ALTER TABLE dim_matches ALTER COLUMN match_id SET NOT NULL")
spark.sql("ALTER TABLE dim_matches ADD CONSTRAINT pk_dim_matches PRIMARY KEY (match_id) RELY")

spark.sql("ALTER TABLE dim_franchises_2027 ALTER COLUMN team_code SET NOT NULL")
spark.sql("ALTER TABLE dim_franchises_2027 ADD CONSTRAINT pk_dim_franchises PRIMARY KEY (team_code) RELY")

spark.sql("ALTER TABLE fact_ball_by_ball ADD CONSTRAINT fk_balls_match FOREIGN KEY (match_id) REFERENCES dim_matches(match_id) RELY")

spark.sql("COMMENT ON TABLE fact_ball_by_ball IS 'Core IPL delivery-level fact table tagged by Powerplay (1-6), Middle (7-15), and Death (16-20) phases.'")
spark.sql("COMMENT ON TABLE dim_franchises_2027 IS 'Official IPL 2027 post-retention franchise purses in INR Crores, open squad slots, and open overseas slots.'")
spark.sql("COMMENT ON TABLE dim_auction_pool_2027 IS 'Released players available in the IPL 2027 mini-auction pool with base and projected auction prices in INR Crores.'")
spark.sql("COMMENT ON VIEW gold_bowler_phase_metrics IS 'Pre-aggregated IPL bowler economy and strike rate by match phase and pitch type, joined to 2027 auction pool prices.'")
spark.sql("COMMENT ON VIEW gold_batter_phase_metrics IS 'Pre-aggregated IPL batter strike rate and boundary counts by match phase and pitch type, joined to 2027 auction pool prices.'")

print("✅ Gold Star Schema Data Mart & Genie Ontology metadata ready in ipl2027.main!")

In [0]:
%sql
SELECT DISTINCT bowler_name 
FROM ipl2027.main.fact_ball_by_ball 
WHERE bowler_name LIKE '%Pathirana%' OR bowler_name LIKE '%Boult%';

In [0]:
from pyspark.sql import functions as F

spark.sql("USE ipl2027.main")

# Drop existing table cleanly to avoid DELTA_METADATA_MISMATCH under Table ACLs
spark.sql("DROP TABLE IF EXISTS dim_auction_pool_2027")

# 1. GOLD DIMENSION: dim_matches
spark.sql("CREATE OR REPLACE TABLE dim_matches AS SELECT * FROM silver_matches")

# 2. GOLD DIMENSION: dim_players
spark.sql("CREATE OR REPLACE TABLE dim_players AS SELECT * FROM silver_players")

# 3. GOLD DIMENSION: dim_franchises_2027 (Official Post-Retention Purses & Open Slots)
franchises_2027 = [
    ("LSG",  "Lucknow Super Giants",        47.45, 18, 7, 3, "Middle-Order Power & Overseas Pace"),
    ("KKR",  "Kolkata Knight Riders",       41.65, 17, 8, 3, "Death Bowler & Top-Order Anchor"),
    ("SRH",  "Sunrisers Hyderabad",         40.15, 18, 7, 3, "Explosive Opener & Overseas All-Rounder"),
    ("DC",   "Delhi Capitals",              39.50, 16, 9, 4, "Left-Arm Pace & Top-Order Batter"),
    ("MI",   "Mumbai Indians",              32.60, 16, 9, 2, "Overseas Spinner & Powerplay Pacer"),
    ("CSK",  "Chennai Super Kings",         28.05, 19, 6, 2, "Indian All-Rounder & Death Pacer"),
    ("RR",   "Rajasthan Royals",            27.05, 19, 6, 3, "Lower-Order Finisher & Left-Arm Pacer"),
    ("PBKS", "Punjab Kings",                18.60, 19, 6, 2, "Backup Overseas Fast Bowler"),
    ("RCB",  "Royal Challengers Bengaluru", 14.15, 20, 5, 1, "Indian Domestic Spin Depth"),
    ("GT",   "Gujarat Titans",               8.75, 22, 3, 2, "Budget Domestic Backup Finisher")
]
df_franchises = spark.createDataFrame(
    franchises_2027,
    ["team_code", "team_name", "remaining_purse_cr", "retained_count", "open_slots", "open_overseas_slots", "primary_auction_need"]
)
df_franchises.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_franchises_2027")

# 4. GOLD DIMENSION: dim_auction_pool_2027 (Mapped to Exact Cricsheet Initials in ball_by_ball)
auction_pool_2027 = [
    ("Matheesha Pathirana", "M Pathirana",     "SL",  True,  "Bowler",      "Death Specialist (Sling)",      "KKR",  2.00, 16.50),
    ("Trent Boult",         "TA Boult",        "NZ",  True,  "Bowler",      "Left-Arm Powerplay Pacer",      "MI",   2.00, 11.00),
    ("Shimron Hetmyer",     "SO Hetmyer",      "WI",  True,  "Batter",      "Left-Hand Finisher",            "RR",   1.50,  9.50),
    ("Pathum Nissanka",     "P Nissanka",      "SL",  True,  "Batter",      "Right-Hand Top-Order",          "DC",   1.50,  8.75),
    ("Rachin Ravindra",     "R Ravindra",      "NZ",  True,  "All-Rounder", "Left-Arm Spin All-Rounder",     "KKR",  1.50,  8.25),
    ("Anrich Nortje",       "A Nortje",        "SA",  True,  "Bowler",      "Express Pace",                  "LSG",  2.00,  8.50),
    ("Wanindu Hasaranga",   "W Hasaranga",     "SL",  True,  "All-Rounder", "Leg-Spin All-Rounder",          "LSG",  2.00,  9.25),
    ("Lockie Ferguson",     "LH Ferguson",     "NZ",  True,  "Bowler",      "Express Pace",                  "PBKS", 2.00,  7.50),
    ("Avesh Khan",          "Avesh Khan",      "IND", False, "Bowler",      "Indian Death/Middle Pacer",     "LSG",  1.00,  8.00),
    ("Deepak Chahar",       "DL Chahar",       "IND", False, "Bowler",      "Indian Powerplay Swing",        "MI",   1.00,  7.25),
    ("Nehal Wadhera",       "N Wadhera",       "IND", False, "Batter",      "Indian Left-Hand Middle-Order", "PBKS", 0.75,  6.50),
    ("Shahrukh Khan",       "M Shahrukh Khan", "IND", False, "Batter",      "Indian Power Finisher",         "GT",   0.75,  6.00),
    ("Mustafizur Rahman",   "Mustafizur Rahman","BAN",True,  "Bowler",      "Left-Arm Cutter Specialist",    "KKR",  1.50,  6.75)
]
df_auction = spark.createDataFrame(
    auction_pool_2027,
    ["full_name", "cricsheet_name", "country", "is_overseas", "role", "skill_style", "prev_team_code", "base_price_cr", "projected_price_cr"]
)
df_auction.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_auction_pool_2027")

# 5. GOLD FACT TABLE: fact_ball_by_ball
spark.sql("CREATE OR REPLACE TABLE fact_ball_by_ball AS SELECT * FROM silver_ball_by_ball")

# 6. GOLD AGGREGATED VIEWS (Joined on cricsheet_name for 100% match rate)
spark.sql("""
CREATE OR REPLACE VIEW gold_bowler_phase_metrics AS
SELECT
    COALESCE(a.full_name, b.bowler_name) AS player_name,
    b.bowler_name AS cricsheet_name,
    CASE WHEN a.cricsheet_name IS NOT NULL THEN TRUE ELSE FALSE END AS is_in_2027_auction_pool,
    a.country,
    a.is_overseas,
    a.skill_style,
    a.prev_team_code,
    a.base_price_cr,
    a.projected_price_cr,
    m.pitch_type,
    b.match_phase,
    SUM(b.is_legal_ball) AS legal_balls_bowled,
    SUM(b.total_runs) AS runs_conceded,
    SUM(b.is_bowler_wicket) AS wickets_taken,
    ROUND(SUM(b.total_runs) * 6.0 / NULLIF(SUM(b.is_legal_ball), 0), 2) AS economy_rate,
    ROUND(SUM(b.is_legal_ball) * 1.0 / NULLIF(SUM(b.is_bowler_wicket), 0), 2) AS bowling_strike_rate
FROM fact_ball_by_ball b
JOIN dim_matches m ON b.match_id = m.match_id
LEFT JOIN dim_auction_pool_2027 a ON b.bowler_name = a.cricsheet_name
GROUP BY ALL
""")

spark.sql("""
CREATE OR REPLACE VIEW gold_batter_phase_metrics AS
SELECT
    COALESCE(a.full_name, b.batter_name) AS player_name,
    b.batter_name AS cricsheet_name,
    CASE WHEN a.cricsheet_name IS NOT NULL THEN TRUE ELSE FALSE END AS is_in_2027_auction_pool,
    a.country,
    a.is_overseas,
    a.skill_style,
    a.prev_team_code,
    a.base_price_cr,
    a.projected_price_cr,
    m.pitch_type,
    b.match_phase,
    SUM(b.is_legal_ball) AS legal_balls_faced,
    SUM(b.batter_runs) AS total_runs_scored,
    ROUND(SUM(b.batter_runs) * 100.0 / NULLIF(SUM(b.is_legal_ball), 0), 2) AS strike_rate,
    SUM(CASE WHEN b.batter_runs IN (4, 6) THEN 1 ELSE 0 END) AS boundary_count
FROM fact_ball_by_ball b
JOIN dim_matches m ON b.match_id = m.match_id
LEFT JOIN dim_auction_pool_2027 a ON b.batter_name = a.cricsheet_name
GROUP BY ALL
""")

# 7. ADD PRIMARY & FOREIGN KEYS + COMMENTS (Genie Ontology Grounding)
spark.sql("ALTER TABLE dim_matches ALTER COLUMN match_id SET NOT NULL")
spark.sql("ALTER TABLE dim_matches ADD CONSTRAINT pk_dim_matches PRIMARY KEY (match_id) RELY")

spark.sql("ALTER TABLE dim_franchises_2027 ALTER COLUMN team_code SET NOT NULL")
spark.sql("ALTER TABLE dim_franchises_2027 ADD CONSTRAINT pk_dim_franchises PRIMARY KEY (team_code) RELY")

spark.sql("ALTER TABLE dim_auction_pool_2027 ALTER COLUMN cricsheet_name SET NOT NULL")
spark.sql("ALTER TABLE dim_auction_pool_2027 ADD CONSTRAINT pk_dim_auction_pool PRIMARY KEY (cricsheet_name) RELY")

spark.sql("ALTER TABLE fact_ball_by_ball ADD CONSTRAINT fk_balls_match FOREIGN KEY (match_id) REFERENCES dim_matches(match_id) RELY")

spark.sql("COMMENT ON TABLE fact_ball_by_ball IS 'Core IPL delivery-level fact table tagged by Powerplay (1-6), Middle (7-15), and Death (16-20) phases.'")
spark.sql("COMMENT ON TABLE dim_franchises_2027 IS 'Official IPL 2027 post-retention franchise purses in INR Crores, open squad slots, and open overseas slots.'")
spark.sql("COMMENT ON TABLE dim_auction_pool_2027 IS 'Released players available in the IPL 2027 mini-auction pool mapped to Cricsheet initials, base price, and projected auction price in INR Crores.'")
spark.sql("COMMENT ON VIEW gold_bowler_phase_metrics IS 'Pre-aggregated IPL bowler economy and strike rate by match phase and pitch type, joined to 2027 auction pool prices.'")
spark.sql("COMMENT ON VIEW gold_batter_phase_metrics IS 'Pre-aggregated IPL batter strike rate and boundary counts by match phase and pitch type, joined to 2027 auction pool prices.'")

print("✅ Gold Star Schema & Views created cleanly!")

In [0]:
%sql
SELECT player_name, match_phase, pitch_type, legal_balls_bowled, wickets_taken, economy_rate, projected_price_cr
FROM ipl2027.main.gold_bowler_phase_metrics
WHERE is_in_2027_auction_pool = TRUE
  AND player_name IN ('Matheesha Pathirana', 'Trent Boult')
ORDER BY player_name, match_phase;

In [0]:
df_b = spark.table("ipl2027.main.bronze_ball_by_ball")
print("Columns:", df_b.columns)

# Show sample rows where a wicket fell
wkt_candidates = [c for c in df_b.columns if "wkt" in c or "wicket" in c or "dismiss" in c or "out" in c]
print("Wicket-related columns:", wkt_candidates)
df_b.select(wkt_candidates).filter(F.coalesce(*[F.col(c).cast("string") for c in wkt_candidates]).isNotNull()).distinct().show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

spark.sql("USE ipl2027.main")

df_aliases_clean = spark.table("silver_team_aliases")
df_balls = spark.table("bronze_ball_by_ball")
b_cols = df_balls.columns

over_col    = next((c for c in b_cols if c in ["over", "over_num", "over_number"]), "over")
batter_col  = next((c for c in b_cols if c in ["batter", "striker", "batsman"]), "batter")
bowler_col  = next((c for c in b_cols if c in ["bowler"]), "bowler")
bat_run_col = next((c for c in b_cols if c in ["batsman_runs", "runs_off_bat", "batter_runs"]), "batsman_runs")
tot_run_col = next((c for c in b_cols if c in ["total_runs", "runs_total"]), bat_run_col)
bat_tm_col  = next((c for c in b_cols if c in ["batting_team", "team_batting"]), "batting_team")
bowl_tm_col = next((c for c in b_cols if c in ["bowling_team", "team_bowling"]), "bowling_team")
wide_col    = next((c for c in b_cols if "wide" in c), None)
noball_col  = next((c for c in b_cols if "noball" in c or "no_ball" in c), None)

# Expanded detection for wicket description column
wkt_col = next(
    (c for c in b_cols if c in ["dismissal_kind", "wicket_type", "wicket_kind", "out_type", "dismissal_type"]),
    None
)
print(f"Detected wicket column: {wkt_col}")

min_over = df_balls.agg(F.min(F.col(over_col).cast("int"))).collect()[0][0]
over_offset = 1 if min_over == 0 else 0

df_balls_silver = (
    df_balls
    .filter(F.col("inning").isin([1, 2]) if "inning" in b_cols else F.col("innings").isin([1, 2]))
    .withColumn("over_num", F.col(over_col).cast("int") + F.lit(over_offset))
    .withColumn(
        "match_phase",
        F.when(F.col("over_num") <= 6, "Powerplay")
         .when(F.col("over_num") <= 15, "Middle")
         .otherwise("Death")
    )
    .withColumn(
        "is_legal_ball",
        F.when(
            (F.coalesce(F.col(wide_col), F.lit(0)) == 0 if wide_col else F.lit(True)) &
            (F.coalesce(F.col(noball_col), F.lit(0)) == 0 if noball_col else F.lit(True)),
            1
        ).otherwise(0)
    )
    .withColumn(
        "is_bowler_wicket",
        F.when(
            F.lower(F.trim(F.col(wkt_col))).isin(
                ["bowled", "caught", "lbw", "stumped", "caught and bowled", "hit wicket"]
            ),
            1
        ).otherwise(0) if wkt_col else F.lit(0)
    )
    .join(df_aliases_clean.alias("bat_a"), F.trim(F.col(bat_tm_col)) == F.col("bat_a.raw_team_name"), "left")
    .join(df_aliases_clean.alias("bowl_a"), F.trim(F.col(bowl_tm_col)) == F.col("bowl_a.raw_team_name"), "left")
    .withColumn("batting_team_code", F.coalesce(F.col("bat_a.standard_team_code"), F.col(bat_tm_col)))
    .withColumn("bowling_team_code", F.coalesce(F.col("bowl_a.standard_team_code"), F.col(bowl_tm_col)))
    .select(
        "match_id",
        "over_num",
        "match_phase",
        "batting_team_code",
        "bowling_team_code",
        F.trim(F.col(batter_col)).alias("batter_name"),
        F.trim(F.col(bowler_col)).alias("bowler_name"),
        F.coalesce(F.col(bat_run_col).cast("int"), F.lit(0)).alias("batter_runs"),
        F.coalesce(F.col(tot_run_col).cast("int"), F.lit(0)).alias("total_runs"),
        "is_legal_ball",
        "is_bowler_wicket"
    )
)

df_balls_silver.write.mode("overwrite").saveAsTable("silver_ball_by_ball")
spark.sql("CREATE OR REPLACE TABLE fact_ball_by_ball AS SELECT * FROM silver_ball_by_ball")

# Re-verify Pathirana & Boult wickets!
spark.sql("""
SELECT player_name, match_phase, pitch_type, legal_balls_bowled, wickets_taken, economy_rate, bowling_strike_rate
FROM ipl2027.main.gold_bowler_phase_metrics
WHERE is_in_2027_auction_pool = TRUE
  AND player_name IN ('Matheesha Pathirana', 'Trent Boult')
ORDER BY player_name, match_phase
""").show()

In [0]:
from pyspark.sql import functions as F

spark.sql("USE ipl2027.main")

df_aliases_clean = spark.table("silver_team_aliases")
df_balls = spark.table("bronze_ball_by_ball")

# Check if over_number starts at 0 or 1 so Powerplay is always 1..6, Middle 7..15, Death 16..20
min_over = df_balls.agg(F.min(F.col("over_number").cast("int"))).collect()[0][0]
over_offset = 1 if min_over == 0 else 0

df_balls_silver = (
    df_balls
    .filter((F.col("innings").isin([1, 2])) & (~F.coalesce(F.col("is_super_over"), F.lit(False))))
    .withColumn("over_num", F.col("over_number").cast("int") + F.lit(over_offset))
    .withColumn(
        "match_phase",
        F.when(F.col("over_num") <= 6, "Powerplay")
         .when(F.col("over_num") <= 15, "Middle")
         .otherwise("Death")
    )
    # BOOLEAN-safe legal ball check (excludes wides & no-balls)
    .withColumn(
        "is_legal_ball",
        F.when(
            (~F.coalesce(F.col("is_wide_ball"), F.lit(False))) &
            (~F.coalesce(F.col("is_no_ball"), F.lit(False))),
            1
        ).otherwise(0)
    )
    # Exact bowler wicket credit (excludes run outs, retired hurt, obstructing the field)
    .withColumn(
        "is_bowler_wicket",
        F.when(
            F.coalesce(F.col("is_wicket"), F.lit(False)) &
            F.lower(F.trim(F.col("wicket_kind"))).isin(
                ["bowled", "caught", "lbw", "stumped", "caught and bowled", "hit wicket"]
            ),
            1
        ).otherwise(0)
    )
    .join(df_aliases_clean.alias("bat_a"), F.trim(F.col("team_batting")) == F.col("bat_a.raw_team_name"), "left")
    .join(df_aliases_clean.alias("bowl_a"), F.trim(F.col("team_bowling")) == F.col("bowl_a.raw_team_name"), "left")
    .withColumn("batting_team_code", F.coalesce(F.col("bat_a.standard_team_code"), F.col("team_batting")))
    .withColumn("bowling_team_code", F.coalesce(F.col("bowl_a.standard_team_code"), F.col("team_bowling")))
    .select(
        "season_id",
        "match_id",
        "innings",
        "over_num",
        "ball_number",
        "match_phase",
        "batting_team_code",
        "bowling_team_code",
        F.trim(F.col("batter")).alias("batter_name"),
        F.trim(F.col("bowler")).alias("bowler_name"),
        F.trim(F.col("batsman_type")).alias("batter_hand"),
        F.trim(F.col("bowler_type")).alias("bowler_style"),
        F.coalesce(F.col("batter_runs").cast("int"), F.lit(0)).alias("batter_runs"),
        F.coalesce(F.col("extras").cast("int"), F.lit(0)).alias("extras"),
        F.coalesce(F.col("total_runs").cast("int"), F.lit(0)).alias("total_runs"),
        "is_legal_ball",
        "is_bowler_wicket",
        F.trim(F.col("wicket_kind")).alias("wicket_kind")
    )
)

df_balls_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_ball_by_ball")
spark.sql("CREATE OR REPLACE TABLE fact_ball_by_ball AS SELECT * FROM silver_ball_by_ball")

# Re-attach Foreign Key constraint on fact_ball_by_ball
spark.sql("ALTER TABLE fact_ball_by_ball ADD CONSTRAINT fk_balls_match FOREIGN KEY (match_id) REFERENCES dim_matches(match_id) RELY")

# Verify Pathirana & Boult now show real wickets and strike rates!
spark.sql("""
SELECT player_name, match_phase, pitch_type, legal_balls_bowled, wickets_taken, economy_rate, bowling_strike_rate
FROM ipl2027.main.gold_bowler_phase_metrics
WHERE is_in_2027_auction_pool = TRUE
  AND player_name IN ('Matheesha Pathirana', 'Trent Boult')
ORDER BY player_name, match_phase
""").show(truncate=False)


In [0]:
%sql
CREATE OR REPLACE VIEW ipl2027.main.gold_batter_vs_bowler_style AS
SELECT
    COALESCE(a.full_name, b.batter_name) AS player_name,
    b.batter_name AS cricsheet_name,
    CASE WHEN a.cricsheet_name IS NOT NULL THEN TRUE ELSE FALSE END AS is_in_2027_auction_pool,
    b.batter_hand,
    b.bowler_style,
    b.match_phase,
    SUM(b.is_legal_ball) AS legal_balls_faced,
    SUM(b.batter_runs) AS runs_scored,
    SUM(b.is_bowler_wicket) AS dismissals,
    ROUND(SUM(b.batter_runs) * 100.0 / NULLIF(SUM(b.is_legal_ball), 0), 2) AS strike_rate
FROM ipl2027.main.fact_ball_by_ball b
LEFT JOIN ipl2027.main.dim_auction_pool_2027 a ON b.batter_name = a.cricsheet_name
GROUP BY ALL;